# Agent vs Agentic — The Spectrum in Code

**Goal:** Build four systems — each more autonomous than the last — to see exactly where "agentic features" end and a "real agent" begins. By the end, you'll never confuse these terms again.

**No API keys required** — all demos use simulated LLM responses to focus on the architecture, not the model.

---

## What You'll Learn

1. The four levels of AI autonomy: static → tool-augmented → agentic workflow → autonomous agent
2. What capabilities each level adds (and what it costs in complexity)
3. How to identify which level a real-world system actually needs
4. Why most enterprise use cases need agentic **features**, not standalone agents
5. How to visualize and compare the behavior of each level on the same task

## Setup

In [ ]:
# Uncomment and run if you need to install dependencies
# !pip install matplotlib numpy

In [ ]:
import json
import time
import random
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from datetime import datetime

# ------------------------------------------------------------------
# Shared mock data: CorpX Q3 metrics
# This simulates what would come from a real database or API.
# All four levels will try to answer questions about this same data.
# ------------------------------------------------------------------
CORPX_DATA = {
    "Q3": {
        "revenue": {"total": 38.4, "yoy_growth": 14.0, "qoq_growth": -5.0,
                    "by_practice": {
                        "Digital & Cloud": 14.1, "Risk & Advisory": 11.2,
                        "Healthcare": 7.8, "Energy": 5.3}},
        "headcount": {"total": 620, "new_hires": 32, "departures": 10,
                      "attrition_rate": 9.0, "utilization": 78},
        "clients": {"active": 78, "new": 10, "satisfaction": 4.5,
                    "nps": 58, "retention": 91},
    },
    "Q2": {
        "revenue": {"total": 40.5, "yoy_growth": 16.0, "qoq_growth": 3.0,
                    "by_practice": {
                        "Digital & Cloud": 13.0, "Risk & Advisory": 12.8,
                        "Healthcare": 8.2, "Energy": 6.5}},
        "headcount": {"total": 598, "new_hires": 26, "departures": 16,
                      "attrition_rate": 11.0, "utilization": 82},
        "clients": {"active": 72, "new": 6, "satisfaction": 4.3,
                    "nps": 54, "retention": 90},
    },
}

print("Setup complete! CorpX data loaded for Q2 and Q3.")
print(f"Revenue, headcount, and client metrics available for both quarters.")

---

## Level 1: Static Prompt (Zero Autonomy)

The simplest possible AI interaction. The user asks a question, the system fills in a template with pre-loaded data, and returns a fixed response. **No tools, no reasoning, no decisions.**

This is what most "AI dashboards" actually are today — a prompt template with variables injected.

In [ ]:
# ------------------------------------------------------------------
# LEVEL 1: Static Prompt
# - No tools, no reasoning, no decisions
# - Data is hardcoded into the prompt template
# - Same question ALWAYS produces the same answer
# - Cannot handle questions outside its template
# ------------------------------------------------------------------

class Level1_StaticPrompt:
    """A static prompt system — the simplest 'AI' possible.
    
    Takes a question, matches it to a template, fills in data.
    No tool use, no reasoning, no autonomy.
    """
    
    def __init__(self, data):
        # All data is loaded at init time — no dynamic retrieval
        self.data = data
        self.trace = []  # Track what the system did for comparison later
    
    def ask(self, question):
        """Answer a question using template matching."""
        self.trace = []  # Reset trace for each question
        q = question.lower()
        
        self.trace.append({"action": "template_match", "input": question})
        
        # Simple keyword matching to pick a template
        if "revenue" in q:
            # Hardcoded response — always Q3, always the same format
            rev = self.data["Q3"]["revenue"]
            answer = (f"Q3 revenue was ${rev['total']}M, "
                      f"a {rev['yoy_growth']}% increase year-over-year.")
        elif "headcount" in q or "employee" in q or "hiring" in q:
            hc = self.data["Q3"]["headcount"]
            answer = (f"Headcount is {hc['total']} employees. "
                      f"We hired {hc['new_hires']} and had {hc['departures']} departures. "
                      f"Attrition: {hc['attrition_rate']}%.")
        elif "client" in q or "satisfaction" in q:
            cl = self.data["Q3"]["clients"]
            answer = (f"{cl['active']} active clients. "
                      f"Satisfaction: {cl['satisfaction']}/5. NPS: {cl['nps']}.")
        else:
            # Cannot handle anything outside the templates
            answer = "I don't have a template for that question."
        
        self.trace.append({"action": "response", "output": answer})
        return answer


# Create and test
level1 = Level1_StaticPrompt(CORPX_DATA)

# These work — they match templates
print("LEVEL 1: STATIC PROMPT")
print("=" * 60)
print(f"\n  Q: 'What is our revenue?'")
print(f"  A: {level1.ask('What is our revenue?')}")

print(f"\n  Q: 'How are employees doing?'")
print(f"  A: {level1.ask('How are employees doing?')}")

# This fails — no template for comparisons
print(f"\n  Q: 'Compare Q3 to Q2 revenue and explain the trend'")
print(f"  A: {level1.ask('Compare Q3 to Q2 revenue and explain the trend')}")

# This also fails — question is phrased differently
print(f"\n  Q: 'How much money did we make?'")
print(f"  A: {level1.ask('How much money did we make?')}")

print(f"\n\n💡 Notice: Level 1 can only answer questions that exactly match")
print(f"   its templates. 'How much money' doesn't trigger 'revenue'.")
print(f"   It can't compare quarters, reason about trends, or handle")
print(f"   anything unexpected. This is NOT agentic — it's a lookup table.")

---

## Level 2: Tool-Augmented (Low Autonomy)

Now we give the system **tools** — functions it can call to retrieve data dynamically. It still follows a fixed decision path (no real reasoning), but it can pull the right data based on the question.

**This is what "agentic features" means in practice** — adding tool access to an existing system. Most RAG-powered dashboards live here.

In [ ]:
# ------------------------------------------------------------------
# LEVEL 2: Tool-Augmented
# - Has tools it can call (functions that retrieve data)
# - Still follows a fixed decision path (if/else routing)
# - Can handle more question variations than Level 1
# - But: only calls ONE tool, no chaining, no reasoning
# ------------------------------------------------------------------

class Level2_ToolAugmented:
    """A tool-augmented system — can call functions but follows fixed logic.
    
    The 'intelligence' is in the routing (which tool to call),
    not in reasoning about the results.
    """
    
    def __init__(self, data):
        self.data = data
        self.trace = []
        # Register tools — functions the system can call
        self.tools = {
            "get_revenue": self._get_revenue,
            "get_headcount": self._get_headcount,
            "get_clients": self._get_clients,
        }
    
    # ---- Tools ----
    def _get_revenue(self, quarter="Q3"):
        """Tool: Retrieve revenue data for a given quarter."""
        if quarter not in self.data:
            return {"error": f"No data for {quarter}"}
        return self.data[quarter]["revenue"]
    
    def _get_headcount(self, quarter="Q3"):
        """Tool: Retrieve headcount data for a given quarter."""
        if quarter not in self.data:
            return {"error": f"No data for {quarter}"}
        return self.data[quarter]["headcount"]
    
    def _get_clients(self, quarter="Q3"):
        """Tool: Retrieve client metrics for a given quarter."""
        if quarter not in self.data:
            return {"error": f"No data for {quarter}"}
        return self.data[quarter]["clients"]
    
    # ---- Routing (the 'brain' — still rule-based) ----
    def _route_to_tool(self, question):
        """Decide which tool to call based on keywords.
        
        This is smarter than Level 1 because it handles synonyms,
        but it's still a fixed decision tree — not reasoning.
        """
        q = question.lower()
        # Broader keyword matching — handles more phrasings
        if any(w in q for w in ["revenue", "money", "financial", "sales", "income", "margin"]):
            return "get_revenue"
        elif any(w in q for w in ["headcount", "employee", "hiring", "attrition", "people", "team", "staff"]):
            return "get_headcount"
        elif any(w in q for w in ["client", "satisfaction", "nps", "customer", "retention"]):
            return "get_clients"
        return None
    
    def ask(self, question):
        """Answer a question by routing to the right tool."""
        self.trace = []
        
        # Step 1: Route to the right tool
        tool_name = self._route_to_tool(question)
        self.trace.append({"action": "route", "tool": tool_name, "input": question})
        
        if tool_name is None:
            self.trace.append({"action": "response", "output": "Could not determine which data to retrieve."})
            return "Could not determine which data to retrieve."
        
        # Step 2: Call the tool
        result = self.tools[tool_name]()
        self.trace.append({"action": "tool_call", "tool": tool_name, "result": result})
        
        # Step 3: Format the response (still template-based)
        if "error" in result:
            answer = f"Error: {result['error']}"
        else:
            # Simple formatting of whatever the tool returned
            answer = f"Here's the data: {json.dumps(result, indent=2)}"
        
        self.trace.append({"action": "response", "output": answer[:100]})
        return answer


# Create and test
level2 = Level2_ToolAugmented(CORPX_DATA)

print("LEVEL 2: TOOL-AUGMENTED")
print("=" * 60)

# Now 'How much money' works — broader keyword matching
print(f"\n  Q: 'How much money did we make?'")
print(f"  A: {level2.ask('How much money did we make?')}")

# This still can't compare quarters — only calls ONE tool for Q3
print(f"\n  Q: 'Compare Q3 to Q2 revenue'")
print(f"  A: {level2.ask('Compare Q3 to Q2 revenue')}")

print(f"\n\n💡 Better than Level 1: handles more phrasings ('money' → revenue tool).")
print(f"   But still limited: can't compare quarters (would need TWO tool calls),")
print(f"   can't reason about trends, and dumps raw JSON instead of a narrative.")
print(f"   This is a system WITH a tool, not a system that REASONS about tools.")

---

## Level 3: Agentic Workflow (Medium Autonomy)

Now the system can **chain multiple tool calls**, **reason about results**, and **make decisions** about what to do next. It follows a structured workflow but has real decision-making within that workflow.

**This is what most people actually need.** An existing application (like a dashboard) with agentic capabilities — multi-step reasoning, tool chaining, and basic self-correction.

In [ ]:
# ------------------------------------------------------------------
# LEVEL 3: Agentic Workflow
# - Chains multiple tool calls based on the question
# - Reasons about results (compares, calculates, synthesizes)
# - Makes decisions mid-workflow (needs more data? which tool next?)
# - Self-corrects on errors (retries with different parameters)
# - But: follows a STRUCTURED workflow, not open-ended goal pursuit
# ------------------------------------------------------------------

class Level3_AgenticWorkflow:
    """An agentic workflow — chains tools, reasons, and self-corrects.
    
    The key difference from Level 2: this system can call MULTIPLE tools,
    reason about intermediate results, and adapt its plan mid-execution.
    """
    
    def __init__(self, data):
        self.data = data
        self.trace = []
        # Same tools as Level 2, but now they accept parameters
        self.tools = {
            "get_revenue": self._get_revenue,
            "get_headcount": self._get_headcount,
            "get_clients": self._get_clients,
            "calculate": self._calculate,  # New: computation tool
        }
        self.context = []  # Accumulated context from tool calls
    
    # ---- Tools (same as Level 2, plus calculate) ----
    def _get_revenue(self, quarter="Q3"):
        if quarter not in self.data:
            return {"error": f"No data for {quarter}"}
        return {"quarter": quarter, "type": "revenue", **self.data[quarter]["revenue"]}
    
    def _get_headcount(self, quarter="Q3"):
        if quarter not in self.data:
            return {"error": f"No data for {quarter}"}
        return {"quarter": quarter, "type": "headcount", **self.data[quarter]["headcount"]}
    
    def _get_clients(self, quarter="Q3"):
        if quarter not in self.data:
            return {"error": f"No data for {quarter}"}
        return {"quarter": quarter, "type": "clients", **self.data[quarter]["clients"]}
    
    def _calculate(self, operation, a, b):
        """Tool: Perform a calculation. Supports growth rate, difference, ratio."""
        if operation == "growth":
            rate = ((a - b) / b) * 100
            return {"operation": "growth", "from": b, "to": a, "rate": round(rate, 1)}
        elif operation == "difference":
            return {"operation": "difference", "a": a, "b": b, "result": round(a - b, 1)}
        return {"error": f"Unknown operation: {operation}"}
    
    # ---- The 'brain': plan, execute, reason ----
    def _analyze_question(self, question):
        """Analyze what the question needs — which tools, which quarters.
        
        This is where 'agentic' reasoning happens: the system figures out
        a multi-step plan based on the question.
        """
        q = question.lower()
        plan = {"tools_needed": [], "quarters": [], "needs_comparison": False}
        
        # Detect which data domains are needed
        if any(w in q for w in ["revenue", "money", "financial", "sales"]):
            plan["tools_needed"].append("get_revenue")
        if any(w in q for w in ["headcount", "employee", "hiring", "attrition", "people", "staff"]):
            plan["tools_needed"].append("get_headcount")
        if any(w in q for w in ["client", "satisfaction", "nps", "customer"]):
            plan["tools_needed"].append("get_clients")
        
        # If no specific domain detected, get everything (broad question)
        if not plan["tools_needed"]:
            plan["tools_needed"] = ["get_revenue", "get_headcount", "get_clients"]
        
        # Detect if comparison is needed
        if any(w in q for w in ["compare", "vs", "versus", "trend", "change", "growth", "q2"]):
            plan["quarters"] = ["Q3", "Q2"]  # Need both quarters
            plan["needs_comparison"] = True
        else:
            plan["quarters"] = ["Q3"]  # Just the latest
        
        return plan
    
    def _synthesize(self, question, results):
        """Synthesize collected data into a narrative answer.
        
        This simulates what an LLM would do: take structured data
        and produce a natural-language summary with insights.
        """
        # Group results by type
        by_type = {}
        for r in results:
            if "error" not in r:
                key = r.get("type") or r.get("operation", "other")
                if key not in by_type:
                    by_type[key] = []
                by_type[key].append(r)
        
        parts = []
        
        # Revenue narrative
        if "revenue" in by_type:
            rev_data = by_type["revenue"]
            q3 = next((r for r in rev_data if r["quarter"] == "Q3"), None)
            q2 = next((r for r in rev_data if r["quarter"] == "Q2"), None)
            if q3:
                parts.append(f"Revenue: Q3 was ${q3['total']}M ({q3['yoy_growth']}% YoY).")
            if q2 and q3:
                change = q3["total"] - q2["total"]
                parts.append(f"That's ${abs(change):.1f}M {'below' if change < 0 else 'above'} Q2's ${q2['total']}M.")
        
        # Headcount narrative
        if "headcount" in by_type:
            hc_data = by_type["headcount"]
            q3 = next((r for r in hc_data if r["quarter"] == "Q3"), None)
            q2 = next((r for r in hc_data if r["quarter"] == "Q2"), None)
            if q3:
                parts.append(f"Headcount: {q3['total']} employees, attrition at {q3['attrition_rate']}%.")
            if q2 and q3:
                attr_change = q3["attrition_rate"] - q2["attrition_rate"]
                direction = "improved" if attr_change < 0 else "worsened"
                parts.append(f"Attrition {direction} from {q2['attrition_rate']}% to {q3['attrition_rate']}%.")
        
        # Client narrative
        if "clients" in by_type:
            cl_data = by_type["clients"]
            q3 = next((r for r in cl_data if r["quarter"] == "Q3"), None)
            q2 = next((r for r in cl_data if r["quarter"] == "Q2"), None)
            if q3:
                parts.append(f"Clients: {q3['active']} active, satisfaction {q3['satisfaction']}/5, NPS {q3['nps']}.")
            if q2 and q3:
                nps_change = q3["nps"] - q2["nps"]
                parts.append(f"NPS {'up' if nps_change > 0 else 'down'} {abs(nps_change)} points from Q2.")
        
        # Growth calculations
        if "growth" in by_type:
            for calc in by_type["growth"]:
                parts.append(f"Growth rate: {calc['rate']}% (from {calc['from']} to {calc['to']}).")
        
        return " ".join(parts) if parts else "No relevant data found."
    
    def ask(self, question):
        """Answer a question using a multi-step agentic workflow."""
        self.trace = []
        self.context = []
        
        # Step 1: PLAN — analyze the question and create an execution plan
        plan = self._analyze_question(question)
        self.trace.append({"action": "plan", "plan": plan})
        
        # Step 2: EXECUTE — call tools according to the plan
        results = []
        for tool_name in plan["tools_needed"]:
            for quarter in plan["quarters"]:
                # Call the tool with the right parameters
                result = self.tools[tool_name](quarter=quarter)
                self.trace.append({"action": "tool_call", "tool": tool_name,
                                   "params": {"quarter": quarter}, "result": result})
                
                # Self-correction: if error, try a different approach
                if "error" in result:
                    self.trace.append({"action": "self_correct",
                                       "reason": f"Error from {tool_name}: {result['error']}",
                                       "decision": "skip and continue"})
                    continue
                
                results.append(result)
        
        # Step 3: CALCULATE — if comparison needed, compute growth rates
        if plan["needs_comparison"] and len(results) >= 2:
            # Find Q3 and Q2 revenue for growth calculation
            q3_rev = next((r for r in results if r.get("type") == "revenue" and r.get("quarter") == "Q3"), None)
            q2_rev = next((r for r in results if r.get("type") == "revenue" and r.get("quarter") == "Q2"), None)
            if q3_rev and q2_rev:
                calc = self._calculate("growth", q3_rev["total"], q2_rev["total"])
                self.trace.append({"action": "tool_call", "tool": "calculate",
                                   "params": {"op": "growth"}, "result": calc})
                results.append(calc)
        
        # Step 4: SYNTHESIZE — combine all results into a narrative
        answer = self._synthesize(question, results)
        self.trace.append({"action": "synthesize", "input_count": len(results), "output": answer[:100]})
        
        return answer


# Create and test
level3 = Level3_AgenticWorkflow(CORPX_DATA)

print("LEVEL 3: AGENTIC WORKFLOW")
print("=" * 60)

# Now it can compare quarters!
print(f"\n  Q: 'Compare Q3 to Q2 revenue'")
answer = level3.ask("Compare Q3 to Q2 revenue")
print(f"  A: {answer}")

# Cross-domain question
print(f"\n  Q: 'How are we doing overall? Compare to last quarter'")
answer = level3.ask("How are we doing overall? Compare to last quarter")
print(f"  A: {answer}")

print(f"\n\n💡 Major leap from Level 2: chains {len(level3.trace)} steps,")
print(f"   pulls data from multiple tools, calculates growth rates,")
print(f"   and synthesizes a narrative answer.")
print(f"   This is what 'agentic features' look like in practice.")

---

## Level 4: Autonomous Agent (High Autonomy)

The full agent. It receives a **goal** (not just a question), creates its own plan, executes autonomously, self-corrects on failure, and can even decide to gather more data than originally requested if it determines the answer needs it.

**Key difference from Level 3:** Level 3 follows a structured workflow (analyze → execute → synthesize). Level 4 runs an open-ended **reasoning loop** — it decides what to do next at every step.

In [ ]:
# ------------------------------------------------------------------
# LEVEL 4: Autonomous Agent
# - Open-ended reasoning loop (not a fixed workflow)
# - Creates its own plan and modifies it mid-execution
# - Decides when it has enough information to answer
# - Self-corrects: retries on failure, gathers more data if needed
# - Includes confidence assessment
# - Has a 'memory' of what it's already tried
# ------------------------------------------------------------------

class Level4_AutonomousAgent:
    """A fully autonomous agent with open-ended reasoning.
    
    The key difference from Level 3: this agent doesn't follow a
    predetermined workflow. It reasons at EVERY step about what
    to do next, and can change its plan based on what it discovers.
    """
    
    def __init__(self, data):
        self.data = data
        self.trace = []
        self.memory = []     # What the agent has learned so far
        self.plan = []       # Current plan (can be modified mid-execution)
        self.tools = {
            "get_revenue": self._get_revenue,
            "get_headcount": self._get_headcount,
            "get_clients": self._get_clients,
            "calculate": self._calculate,
            "assess_risk": self._assess_risk,  # New: risk assessment tool
        }
    
    # ---- Tools ----
    def _get_revenue(self, quarter="Q3"):
        if quarter not in self.data:
            return {"error": f"No data for {quarter}"}
        return {"quarter": quarter, "type": "revenue", **self.data[quarter]["revenue"]}
    
    def _get_headcount(self, quarter="Q3"):
        if quarter not in self.data:
            return {"error": f"No data for {quarter}"}
        return {"quarter": quarter, "type": "headcount", **self.data[quarter]["headcount"]}
    
    def _get_clients(self, quarter="Q3"):
        if quarter not in self.data:
            return {"error": f"No data for {quarter}"}
        return {"quarter": quarter, "type": "clients", **self.data[quarter]["clients"]}
    
    def _calculate(self, operation="growth", a=0, b=0):
        if operation == "growth" and b != 0:
            return {"operation": "growth", "from": b, "to": a, "rate": round(((a-b)/b)*100, 1)}
        elif operation == "difference":
            return {"operation": "difference", "result": round(a - b, 1)}
        return {"error": "Invalid calculation"}
    
    def _assess_risk(self, metric, value, threshold):
        """Tool: Assess if a metric is at risk based on a threshold."""
        status = "healthy" if value >= threshold else "at_risk"
        gap = round(value - threshold, 1)
        return {"metric": metric, "value": value, "threshold": threshold,
                "status": status, "gap": gap}
    
    # ---- The reasoning engine ----
    def _think(self, goal, step_num):
        """The agent's reasoning step: what should I do next?
        
        In a real agent, this would be an LLM call. Here we simulate
        the REASONING PATTERN that an LLM would follow.
        """
        # What data do I already have?
        collected_types = [m["type"] for m in self.memory if "type" in m]
        collected_quarters = [m.get("quarter") for m in self.memory if "quarter" in m]
        
        g = goal.lower()
        
        # First: figure out what data I need but don't have yet
        needs_revenue = any(w in g for w in ["revenue", "financial", "money", "sales", "overall", "executive"])
        needs_headcount = any(w in g for w in ["headcount", "employee", "hiring", "people", "overall", "executive"])
        needs_clients = any(w in g for w in ["client", "satisfaction", "nps", "overall", "executive"])
        needs_comparison = any(w in g for w in ["compare", "trend", "change", "growth", "q2", "quarter"])
        
        # Decision: what to do next
        if needs_revenue and "revenue" not in collected_types:
            return {"action": "tool_call", "tool": "get_revenue", "params": {"quarter": "Q3"},
                    "reasoning": "I need revenue data to answer this question."}
        
        if needs_revenue and needs_comparison and ("Q2" not in collected_quarters or
                                                    not any(m.get("type") == "revenue" and m.get("quarter") == "Q2" for m in self.memory)):
            return {"action": "tool_call", "tool": "get_revenue", "params": {"quarter": "Q2"},
                    "reasoning": "Need Q2 revenue data for comparison."}
        
        if needs_headcount and "headcount" not in collected_types:
            return {"action": "tool_call", "tool": "get_headcount", "params": {"quarter": "Q3"},
                    "reasoning": "I need headcount data."}
        
        if needs_headcount and needs_comparison and not any(m.get("type") == "headcount" and m.get("quarter") == "Q2" for m in self.memory):
            return {"action": "tool_call", "tool": "get_headcount", "params": {"quarter": "Q2"},
                    "reasoning": "Need Q2 headcount for comparison."}
        
        if needs_clients and "clients" not in collected_types:
            return {"action": "tool_call", "tool": "get_clients", "params": {"quarter": "Q3"},
                    "reasoning": "I need client metrics."}
        
        if needs_clients and needs_comparison and not any(m.get("type") == "clients" and m.get("quarter") == "Q2" for m in self.memory):
            return {"action": "tool_call", "tool": "get_clients", "params": {"quarter": "Q2"},
                    "reasoning": "Need Q2 client data for comparison."}
        
        # --- AUTONOMOUS BEHAVIOR: detect risks even if not asked ---
        # This is what makes it an AGENT: it goes beyond the literal question
        rev_data = [m for m in self.memory if m.get("type") == "revenue" and m.get("quarter") == "Q3"]
        hc_data = [m for m in self.memory if m.get("type") == "headcount" and m.get("quarter") == "Q3"]
        risk_assessed = any(m.get("action") == "risk_check" for m in self.memory)
        
        if rev_data and not risk_assessed:
            qoq = rev_data[0].get("qoq_growth", 0)
            if qoq < 0:
                return {"action": "tool_call", "tool": "assess_risk",
                        "params": {"metric": "qoq_revenue_growth", "value": qoq, "threshold": 0},
                        "reasoning": f"I noticed QoQ growth is {qoq}% — this is negative. Let me flag this as a risk."}
        
        if hc_data and not risk_assessed:
            util = hc_data[0].get("utilization", 100)
            if util < 80:
                return {"action": "tool_call", "tool": "assess_risk",
                        "params": {"metric": "utilization", "value": util, "threshold": 80},
                        "reasoning": f"Utilization at {util}% is below target. Flagging as a risk."}
        
        # If we've collected enough, synthesize
        return {"action": "answer", "reasoning": "I have enough data to provide a comprehensive answer."}
    
    def _synthesize_answer(self, goal):
        """Produce a final answer from everything in memory."""
        parts = []
        risks = []
        
        # Revenue summary
        q3_rev = next((m for m in self.memory if m.get("type") == "revenue" and m.get("quarter") == "Q3"), None)
        q2_rev = next((m for m in self.memory if m.get("type") == "revenue" and m.get("quarter") == "Q2"), None)
        if q3_rev:
            parts.append(f"Revenue: ${q3_rev['total']}M in Q3 ({q3_rev['yoy_growth']}% YoY).")
        if q2_rev and q3_rev:
            change = q3_rev['total'] - q2_rev['total']
            parts.append(f"QoQ: ${change:+.1f}M vs Q2 (${q2_rev['total']}M).")
        
        # Headcount summary
        q3_hc = next((m for m in self.memory if m.get("type") == "headcount" and m.get("quarter") == "Q3"), None)
        q2_hc = next((m for m in self.memory if m.get("type") == "headcount" and m.get("quarter") == "Q2"), None)
        if q3_hc:
            parts.append(f"Headcount: {q3_hc['total']} (attrition {q3_hc['attrition_rate']}%, utilization {q3_hc['utilization']}%).")
        if q2_hc and q3_hc:
            parts.append(f"Attrition improved from {q2_hc['attrition_rate']}% to {q3_hc['attrition_rate']}%.")
        
        # Client summary
        q3_cl = next((m for m in self.memory if m.get("type") == "clients" and m.get("quarter") == "Q3"), None)
        if q3_cl:
            parts.append(f"Clients: {q3_cl['active']} active, NPS {q3_cl['nps']}, satisfaction {q3_cl['satisfaction']}/5.")
        
        # Risk flags (AUTONOMOUS — agent adds these on its own)
        risk_results = [m for m in self.memory if m.get("action") == "risk_check"]
        for risk in risk_results:
            if risk.get("status") == "at_risk":
                risks.append(f"\u26a0\ufe0f RISK: {risk['metric']} is {risk['value']} (threshold: {risk['threshold']})")
        
        answer = " ".join(parts)
        if risks:
            answer += "\n\n" + "\n".join(risks)
        
        return answer
    
    def run(self, goal, max_steps=10):
        """Run the autonomous agent loop."""
        self.trace = []
        self.memory = []
        
        print(f"\n{'='*60}")
        print(f"\U0001f916 AGENT received goal: \"{goal}\"")
        print(f"{'='*60}")
        
        for step in range(max_steps):
            # THINK: What should I do next?
            decision = self._think(goal, step)
            reasoning = decision.get("reasoning", "")
            
            print(f"\n  Step {step + 1}: [THINK] {reasoning}")
            self.trace.append({"action": "think", "reasoning": reasoning, "decision": decision["action"]})
            
            if decision["action"] == "answer":
                # Generate final answer
                answer = self._synthesize_answer(goal)
                print(f"\n  \U0001f4ac FINAL ANSWER:")
                for line in answer.split("\n"):
                    print(f"     {line}")
                self.trace.append({"action": "answer", "output": answer})
                return answer
            
            elif decision["action"] == "tool_call":
                tool_name = decision["tool"]
                params = decision["params"]
                
                print(f"          [ACT] Calling {tool_name}({json.dumps(params)})")
                
                # Execute the tool
                result = self.tools[tool_name](**params)
                
                # OBSERVE: process the result
                if "error" in result:
                    print(f"          [OBSERVE] Error: {result['error']}")
                    print(f"          [ADAPT] Will skip this and try something else.")
                    self.trace.append({"action": "error", "tool": tool_name, "error": result["error"]})
                else:
                    # Store in memory for later synthesis
                    if tool_name == "assess_risk":
                        result["action"] = "risk_check"
                    self.memory.append(result)
                    print(f"          [OBSERVE] Got {result.get('type', result.get('operation', 'result'))} data. Stored in memory.")
                    self.trace.append({"action": "tool_call", "tool": tool_name, "result": "success"})
        
        return "Agent reached step limit."


# Create and test
level4 = Level4_AutonomousAgent(CORPX_DATA)

print("LEVEL 4: AUTONOMOUS AGENT")
print("=" * 60)
level4.run("Give me an executive overview comparing Q3 to Q2. Flag any risks.")

In [ ]:
# ------------------------------------------------------------------
# Now watch the agent do something AUTONOMOUS:
# Even though the question is just about revenue, the agent notices
# a QoQ decline and proactively flags it as a risk — going BEYOND
# what was asked. This is what separates an agent from agentic features.
# ------------------------------------------------------------------

print("\n" + "*" * 60)
print("AUTONOMOUS BEHAVIOR DEMO: The agent goes beyond the question")
print("*" * 60)

level4_demo = Level4_AutonomousAgent(CORPX_DATA)
level4_demo.run("What's our revenue this quarter?")

print(f"\n\n\U0001f4a1 Notice: The user only asked about revenue.")
print(f"   Level 1 would return a number. Level 2 would return JSON.")
print(f"   Level 3 would give a revenue narrative.")
print(f"   Level 4 ALSO flagged the QoQ decline as a risk — without being asked.")
print(f"   That autonomous initiative is what makes it an AGENT, not just agentic.")

---

## Side-by-Side Comparison: Same Question, Four Levels

Let's run the exact same question through all four levels and compare what happens.

In [ ]:
# ------------------------------------------------------------------
# Run the same question through all four levels and compare
# This makes the differences concrete and visual
# ------------------------------------------------------------------

test_question = "How much money did we make and how does it compare to last quarter?"

print("=" * 70)
print(f"  SAME QUESTION, FOUR LEVELS")
print(f"  Q: \"{test_question}\"")
print("=" * 70)

# Level 1: Static Prompt
l1 = Level1_StaticPrompt(CORPX_DATA)
a1 = l1.ask(test_question)
print(f"\n\U0001f535 LEVEL 1 — Static Prompt (zero autonomy)")
print(f"  Steps: {len(l1.trace)}")
print(f"  Answer: {a1}")

# Level 2: Tool-Augmented
l2 = Level2_ToolAugmented(CORPX_DATA)
a2 = l2.ask(test_question)
print(f"\n\U0001f7e1 LEVEL 2 — Tool-Augmented (low autonomy)")
print(f"  Steps: {len(l2.trace)}")
# Truncate the JSON dump for readability
a2_short = a2[:120] + "..." if len(a2) > 120 else a2
print(f"  Answer: {a2_short}")

# Level 3: Agentic Workflow
l3 = Level3_AgenticWorkflow(CORPX_DATA)
a3 = l3.ask(test_question)
print(f"\n\U0001f7e3 LEVEL 3 — Agentic Workflow (medium autonomy)")
print(f"  Steps: {len(l3.trace)}")
print(f"  Answer: {a3}")

# Level 4: Autonomous Agent
l4 = Level4_AutonomousAgent(CORPX_DATA)
# Suppress the agent's verbose output for the comparison
import io, sys
old_stdout = sys.stdout
sys.stdout = io.StringIO()  # Capture output
a4 = l4.run(test_question)
sys.stdout = old_stdout  # Restore output
print(f"\n\U0001f534 LEVEL 4 — Autonomous Agent (high autonomy)")
print(f"  Steps: {len(l4.trace)}")
print(f"  Answer: {a4}")

print(f"\n\n{'='*70}")
print(f"\U0001f4a1 KEY OBSERVATIONS:")
print(f"  Level 1: Failed — 'money' didn't match any template")
print(f"  Level 2: Got revenue data but couldn't compare to Q2")
print(f"  Level 3: Compared Q3 vs Q2 with growth calculation")
print(f"  Level 4: Did everything Level 3 did PLUS flagged QoQ risk")

---

## Visualizing the Spectrum

In [ ]:
# ------------------------------------------------------------------
# Visualization 1: Capability comparison across all four levels
# Shows what each level CAN and CANNOT do
# ------------------------------------------------------------------

# Define capabilities and which levels have them
capabilities = [
    "Answer basic questions",
    "Handle synonym phrasings",
    "Call tools dynamically",
    "Chain multiple tool calls",
    "Compare across time periods",
    "Calculate derived metrics",
    "Synthesize a narrative",
    "Self-correct on errors",
    "Proactively flag risks",
    "Go beyond the question",
]

# 1 = has capability, 0 = doesn't
levels_data = {
    "L1: Static": [1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    "L2: Tool-Aug": [1, 1, 1, 0, 0, 0, 0, 0, 0, 0],
    "L3: Agentic": [1, 1, 1, 1, 1, 1, 1, 1, 0, 0],
    "L4: Agent": [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
}

fig, ax = plt.subplots(figsize=(12, 6))

# Create the heatmap
data_matrix = np.array(list(levels_data.values()))
colors = np.where(data_matrix == 1, '#2E86C1', '#f0f0f0')  # Blue for yes, gray for no

for i in range(len(levels_data)):
    for j in range(len(capabilities)):
        # Draw colored rectangle
        rect = plt.Rectangle((j - 0.4, i - 0.4), 0.8, 0.8,
                             facecolor=colors[i][j], edgecolor='white', linewidth=2)
        ax.add_patch(rect)
        # Add checkmark or X
        symbol = '\u2713' if data_matrix[i][j] == 1 else ''
        ax.text(j, i, symbol, ha='center', va='center',
                fontsize=14, fontweight='bold',
                color='white' if data_matrix[i][j] == 1 else '#ccc')

ax.set_xlim(-0.5, len(capabilities) - 0.5)
ax.set_ylim(-0.5, len(levels_data) - 0.5)
ax.set_xticks(range(len(capabilities)))
ax.set_yticks(range(len(levels_data)))
ax.set_xticklabels(capabilities, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(list(levels_data.keys()), fontsize=11, fontweight='bold')
ax.set_title('Capability Matrix: Agent vs Agentic Spectrum', fontsize=14, fontweight='bold')

# Add a dividing line between "agentic features" and "agent capabilities"
ax.axvline(x=7.5, color='red', linewidth=2, linestyle='--', alpha=0.7)
ax.text(6.5, -0.8, '\u2190 Agentic Features', ha='right', fontsize=9, color='#666', style='italic')
ax.text(8.5, -0.8, 'Agent Capabilities \u2192', ha='left', fontsize=9, color='#666', style='italic')

plt.tight_layout()
plt.show()

print("\n\U0001f4a1 The red dashed line marks where 'agentic features' end")
print("   and 'autonomous agent' capabilities begin.")
print("   Most enterprise dashboards need everything LEFT of the line.")

In [ ]:
# ------------------------------------------------------------------
# Visualization 2: Execution trace comparison
# Shows HOW each level processes the same question — step by step
# ------------------------------------------------------------------

fig, axes = plt.subplots(1, 4, figsize=(16, 6), sharey=False)

# Color mapping for action types
action_colors = {
    "template_match": "#f0f4ff",   # Light blue — simple lookup
    "route": "#fff3cd",            # Yellow — routing decision
    "tool_call": "#e2d5f1",        # Purple — tool execution
    "plan": "#d4edda",             # Green — planning
    "think": "#d4edda",            # Green — reasoning
    "self_correct": "#f8d7da",     # Red — error handling
    "synthesize": "#cce5ff",       # Blue — narrative generation
    "response": "#f0f4ff",         # Light blue — output
    "answer": "#f0f4ff",           # Light blue — output
    "error": "#f8d7da",            # Red — error
}

# Collect traces from the same question
traces = [
    ("L1: Static", l1.trace),
    ("L2: Tool-Aug", l2.trace),
    ("L3: Agentic", l3.trace),
    ("L4: Agent", l4.trace),
]

for idx, (title, trace) in enumerate(traces):
    ax = axes[idx]
    ax.set_title(title, fontsize=11, fontweight='bold')
    
    # Draw each step as a colored block
    for i, step in enumerate(trace):
        action = step.get("action", "other")
        color = action_colors.get(action, "#f0f0f0")
        
        # Draw the block
        rect = mpatches.FancyBboxPatch(
            (0.1, len(trace) - i - 1), 0.8, 0.8,
            boxstyle="round,pad=0.05",
            facecolor=color, edgecolor='#333', linewidth=1
        )
        ax.add_patch(rect)
        
        # Label the block
        label = action
        if "tool" in step:
            label = step["tool"].replace("get_", "")
        ax.text(0.5, len(trace) - i - 0.6, label, ha='center', va='center',
                fontsize=7, fontweight='bold', color='#333')
    
    ax.set_xlim(0, 1)
    ax.set_ylim(-0.5, max(len(trace), 1) + 0.5)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel(f"{len(trace)} steps", fontsize=9, color='#666')

# Add legend
legend_items = [
    mpatches.Patch(facecolor='#d4edda', edgecolor='#333', label='Plan/Think'),
    mpatches.Patch(facecolor='#e2d5f1', edgecolor='#333', label='Tool Call'),
    mpatches.Patch(facecolor='#cce5ff', edgecolor='#333', label='Synthesize'),
    mpatches.Patch(facecolor='#f8d7da', edgecolor='#333', label='Error/Correct'),
    mpatches.Patch(facecolor='#f0f4ff', edgecolor='#333', label='Response'),
]
fig.legend(handles=legend_items, loc='lower center', ncol=5, fontsize=9,
           bbox_to_anchor=(0.5, -0.02))

fig.suptitle(f'Execution Trace: "{test_question[:50]}..."', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.subplots_adjust(bottom=0.12)
plt.show()

print("\n\U0001f4a1 Each column shows the steps that level takes for the SAME question.")
print("   More steps \u2260 better. The right number of steps depends on the task.")
print("   Level 4 takes more steps because it proactively investigates risks.")

In [ ]:
# ------------------------------------------------------------------
# Visualization 3: Complexity vs Capability tradeoff
# The key insight: more autonomy = more complexity = more risk
# ------------------------------------------------------------------

fig, ax = plt.subplots(figsize=(10, 7))

# Data points for each level
levels = ['L1: Static\nPrompt', 'L2: Tool-\nAugmented', 'L3: Agentic\nWorkflow', 'L4: Autonomous\nAgent']
autonomy = [1, 3, 6, 9]        # X: how much autonomy
complexity = [1, 3, 6, 9]       # Y: how complex to build/maintain
capability = [200, 500, 900, 1400]  # Bubble size: how much it can do
colors_plot = ['#2E86C1', '#ffc107', '#6f42c1', '#dc3545']

# Plot bubbles
for i, (x, y, s, c, label) in enumerate(zip(autonomy, complexity, capability, colors_plot, levels)):
    ax.scatter(x, y, s=s, c=c, alpha=0.7, edgecolors='white', linewidth=2, zorder=3)
    ax.annotate(label, (x, y), textcoords='offset points',
                xytext=(0, -int(s**0.5)/2 - 15), ha='center', fontsize=9, fontweight='bold')

# Add the "sweet spot" annotation
ax.annotate('Sweet spot for\nmost enterprises',
            xy=(6, 6), xytext=(8, 3.5),
            fontsize=10, fontstyle='italic', color='#6f42c1',
            arrowprops=dict(arrowstyle='->', color='#6f42c1', lw=1.5),
            ha='center')

# Shaded region for "what most teams need"
rect = mpatches.FancyBboxPatch((2, 2), 5.5, 5.5, boxstyle="round,pad=0.3",
                                facecolor='#6f42c1', alpha=0.07, edgecolor='#6f42c1',
                                linewidth=1.5, linestyle='--')
ax.add_patch(rect)

ax.set_xlabel('Autonomy Level \u2192', fontsize=12)
ax.set_ylabel('Implementation Complexity \u2192', fontsize=12)
ax.set_title('The Autonomy-Complexity Tradeoff', fontsize=14, fontweight='bold')
ax.set_xlim(0, 10.5)
ax.set_ylim(0, 10.5)
ax.grid(True, alpha=0.2)

# Annotation explaining bubble size
ax.text(0.5, 10, 'Bubble size = capability', fontsize=8, color='#666', style='italic')

plt.tight_layout()
plt.show()

print("\n\U0001f4a1 The purple dashed box is where most enterprise teams should aim.")
print("   Level 3 (Agentic Workflow) gives 80% of the capability")
print("   at a fraction of Level 4's complexity and risk.")
print("\n   Only build a full Level 4 agent when you have:")
print("   - Complex multi-system workflows")
print("   - Clear guardrails and human-in-the-loop")
print("   - Mature monitoring and observability")

---

## Quick Reference: When to Use Which Level

| Scenario | Level | Why |
|---|---|---|
| FAQ bot with canned answers | **L1** Static | Questions are predictable, answers are fixed |
| Dashboard with AI Q&A (RAG) | **L2-L3** Tool-Augmented / Agentic | Needs tool access and maybe multi-step, but within a structured app |
| Report generation from multiple sources | **L3** Agentic Workflow | Chains tools, compares data, synthesizes — but follows a known workflow |
| Client onboarding across CRM + HR + billing | **L4** Agent | Multi-system, unpredictable steps, requires real-time decisions |
| IT incident response | **L4** Agent | Investigates, diagnoses, proposes fixes — each incident is different |
| Regulatory compliance check | **L3** Agentic Workflow | Known checklist, tool-augmented, but structured and auditable |

## Key Takeaways

1. **"Agentic" is a spectrum, not a switch.** Adding one tool to a chatbot makes it slightly agentic. Building a full autonomous agent is the far end of the spectrum. Most useful systems live in the middle.

2. **Most enterprises need Level 3, not Level 4.** Agentic workflows (multi-step, tool-chaining, self-correcting) solve 80% of real business problems. Full autonomous agents add complexity and risk that's rarely justified.

3. **The difference between agentic and agent is INITIATIVE.** An agentic system does what you ask, using tools. An agent goes beyond what you ask — it notices risks, suggests actions, and makes decisions on its own.

4. **More autonomy = more guardrails needed.** Level 1 can't do anything wrong. Level 4 can take actions you didn't expect. The governance requirements scale with the autonomy level.

5. **Start at the lowest level that solves the problem.** Don't build an autonomous agent when a tool-augmented prompt would do. Complexity is a cost — pay it only when you must.

---

### Related Content

- **[Agentic AI Fundamentals](../docs/07-agentic-ai.md)** — The conceptual foundation, including the Agent vs Agentic section
- **[Build a Simple Agent (notebook 06)](06-build-a-simple-agent.ipynb)** — Build a real ReAct agent with actual LLM calls
- **[Agent Orchestration Patterns](../docs/09-orchestration-patterns.md)** — Multi-agent architectures for Level 4 systems
- **[AI Dashboard Architecture](../docs/14-ai-dashboard-architecture.md)** — A Level 2-3 system in production: React dashboard with agentic Q&A